In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        B, T, C = x.size()
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)
        # reshape to (B, n_head, T, head_dim)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        # efficient attention with built-in causal mask
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=self.dropout.p if self.training else 0.0)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y))


class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc   = nn.Linear(config.n_embd, 4 * config.n_embd)
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd)
        self.act    = nn.GELU()
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        return self.dropout(self.c_proj(self.act(self.c_fc(x))))


class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn  = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp   = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))   # pre-norm residual
        x = x + self.mlp(self.ln_2(x))
        return x


class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            drop = nn.Dropout(config.dropout),
            blocks = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = nn.LayerNorm(config.n_embd),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        # weight tying
        self.transformer.wte.weight = self.lm_head.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.size()
        pos = torch.arange(0, T, device=idx.device)
        x = self.transformer.drop(
            self.transformer.wte(idx) + self.transformer.wpe(pos)
        )
        for block in self.transformer.blocks:
            x = block(x)
        x = self.transformer.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=0.8, top_k=50):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.config.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = -float('Inf')
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, idx_next], dim=1)
        return idx

In [ ]:
print(torch.cuda.is_available())       # should be True
print(torch.cuda.get_device_name(0))   # should show your GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

True
NVIDIA A100-SXM4-40GB


In [ ]:
from dataclasses import dataclass
import numpy as np

epochs = 6

@dataclass
class GPTConfig:
    vocab_size: int = 32000
    block_size: int = 256       # shorter context saves memory
    n_layer: int = 4
    n_head: int = 4
    n_embd: int = 128
    dropout: float = 0.2        # higher dropout for small data


# Pre-load everything into a single tensor on device (2M tokens fits easily in memory)
data_tensor = torch.tensor(np.load("/content/token_ids.npy"), dtype=torch.long, device=device)

class TextDataset(torch.utils.data.Dataset):
    def __init__(self, data, block_size):
        self.data = data  # already on GPU
        self.block_size = block_size

    def __len__(self):
        return len(self.data) - self.block_size

    def __getitem__(self, i):
        return self.data[i:i+self.block_size], self.data[i+1:i+self.block_size+1]

# --- Training ---


def train():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    config = GPTConfig()
    model = GPT(config).to(device)

    # load your pre-tokenized corpus (from the tokenization step)
    data = np.load("/content/token_ids.npy").astype(np.int64)
    split = int(0.9 * len(data))
    train_ds = TextDataset(data[:split], config.block_size)
    val_ds   = TextDataset(data[split:], config.block_size)

    train_loader = torch.utils.data.DataLoader(
        train_ds, batch_size=256, shuffle=True, num_workers=4, pin_memory=True
    )
    val_loader = torch.utils.data.DataLoader(
        val_ds, batch_size=256, shuffle=False
    )

    optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.1)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=len(train_loader) * epochs  # 10 epochs
    )

    # optional: mixed precision
    scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for step, (x, y) in enumerate(train_loader):
            x, y = x.to(device), y.to(device)

            with torch.amp.autocast("cuda", enabled=(device == "cuda")):
                _, loss = model(x, y)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            total_loss += loss.item()

            if step % 100 == 0:
                print(f"epoch {epoch} step {step} loss {loss.item():.4f}")

        # validation
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                _, loss = model(x, y)
                val_loss += loss.item()
        val_loss /= len(val_loader)
        print(f"epoch {epoch} | train {total_loss/len(train_loader):.4f} | val {val_loss:.4f}")

    torch.save(model.state_dict(), "gpt_model.pt")

In [ ]:
train()

epoch 0 step 0 loss 10.3959
epoch 0 step 100 loss 9.2265
epoch 0 step 200 loss 8.3475
epoch 0 step 300 loss 7.6589
epoch 0 step 400 loss 7.1637
epoch 0 step 500 loss 6.8143
epoch 0 step 600 loss 6.5368
epoch 0 step 700 loss 6.2795
epoch 0 step 800 loss 6.1321
epoch 0 step 900 loss 5.9346
epoch 0 step 1000 loss 5.7745
epoch 0 step 1100 loss 5.6702
epoch 0 step 1200 loss 5.5223
epoch 0 step 1300 loss 5.5125
epoch 0 step 1400 loss 5.3702
epoch 0 step 1500 loss 5.2934
epoch 0 step 1600 loss 5.2778
epoch 0 step 1700 loss 5.1877
epoch 0 step 1800 loss 5.1764
epoch 0 step 1900 loss 5.1358
epoch 0 step 2000 loss 5.0903
epoch 0 step 2100 loss 5.0540
epoch 0 step 2200 loss 5.0494
epoch 0 step 2300 loss 5.0015
epoch 0 step 2400 loss 4.9797
epoch 0 step 2500 loss 5.0058
epoch 0 step 2600 loss 4.9730
epoch 0 step 2700 loss 4.8962
epoch 0 step 2800 loss 4.9017
epoch 0 step 2900 loss 4.8988
epoch 0 step 3000 loss 4.8428
epoch 0 step 3100 loss 4.8352
epoch 0 step 3200 loss 4.8763
epoch 0 step 3300 los

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

# 1. Initialize a BPE tokenizer
tokenizer = Tokenizer(models.BPE())
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tokenizer.decoder = decoders.ByteLevel()

# 2. Train on your corpus
trainer = trainers.BpeTrainer(
    vocab_size=32000,          # typical range: 8k–64k
    special_tokens=["<pad>", "<s>", "</s>", "<unk>", "<mask>"],
    min_frequency=2,
)
tokenizer.train(files=["family_guy_corpus.txt"], trainer=trainer)
tokenizer.save("tokenizer.json")

In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# Load and generate
model = GPT(GPTConfig()).to(device)
model.load_state_dict(torch.load("/content/gpt_model.pt"))
model.eval()

prompt = "Hey quagmire"
ids = tokenizer.encode(prompt).ids
x = torch.tensor([ids], dtype=torch.long, device=device)
out = model.generate(x, max_new_tokens=200, temperature=0.8, top_k=50)
print(tokenizer.decode(out[0].tolist()))

Hey quagmire.Let's go check in and get out of here.- Hi, Jillian.- I love you.I'm going to New York.- It's like you're not allowed to be home.- But, this is not the area.I don't know how to say anything.What's your name, Brian? Come on, let's go.Hey, we should go to the park.Look, there's a thing to do with my dad.That's a weird thing.Like the time I discovered that, uh, um, oh, hey, hey, uh, I want to talk.How is it? What do you like? I am.You know, I'm gonna go out with you.I kind of got him to take him back, and then you don't think so.Oh, my God! Oh, oh, yeah? What are you doing with me? I don't know, I just I know.I'm sorry.I didn't
